# Dataset
Many different collections of code/commit related datasets available here: https://huggingface.co/collections/smcleish/diff-datasets

Chose to use CommitPackFT, and only the Python split for this project for consistency: https://huggingface.co/datasets/bigcode/commitpackft

In [ ]:
# Pre-requisites
!pip install -U "datasets<3.0.0"
!pip install -q pandas tqdm
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
# Load dataset
from datasets import load_dataset
import pandas as pd
import difflib
from tqdm import tqdm

# Load only the Python subset
ds = load_dataset("bigcode/commitpackft", "python", split="train", trust_remote_code=True)

print(ds)
print(ds.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/56025 [00:00<?, ? examples/s]

Dataset({
    features: ['commit', 'old_file', 'new_file', 'old_contents', 'new_contents', 'subject', 'message', 'lang', 'license', 'repos'],
    num_rows: 56025
})
['commit', 'old_file', 'new_file', 'old_contents', 'new_contents', 'subject', 'message', 'lang', 'license', 'repos']


In [ ]:
# Convert to DataFrame + Inspect
df = ds.to_pandas()

print("Shape:", df.shape)
display(df.head())

print("\nMissing values:")
display(df.isnull().sum())

print("\nSample commit subjects:")
display(df["subject"].head(10))

Shape: (56025, 10)


,commit,old_file,new_file,old_contents,new_contents,subject,message,lang,license,repos
0,e905334869af72025592de586b81650cb3468b8a,sentry/queue/client.py,sentry/queue/client.py,"""""""\nsentry.queue.client\n~~~~~~~~~~~~~~~~~~~\...","""""""\nsentry.queue.client\n~~~~~~~~~~~~~~~~~~~\...",Declare queues when broker is instantiated,Declare queues when broker is instantiated\n,Python,bsd-3-clause,"imankulov/sentry,BuildingLink/sentry,zenefits/..."
1,45fc612fdc5a354dbf0bacccd345b1aebcc73e59,tests/test_openweather.py,tests/test_openweather.py,# -*- coding: utf-8 -*-\nimport bot_mock\nfrom...,# -*- coding: utf-8 -*-\nimport bot_mock\nfrom...,"Revert ""Fix openweather unit tests""","Revert ""Fix openweather unit tests""\n\nThis re...",Python,bsd-3-clause,"rnyberg/pyfibot,EArmour/pyfibot,aapa/pyfibot,a..."
2,22faee82e1f070532c0dfe5777136e842233a1f0,src/dashboard/src/main/templatetags/percentage.py,src/dashboard/src/main/templatetags/percentage.py,"from django.template import Node, Library\n\nr...","from django.template import Node, Library\n\nr...","Fix % only showing 0 or 100%, everything betwe...","Fix % only showing 0 or 100%, everything betwe...",Python,agpl-3.0,"artefactual/archivematica-history,artefactual/..."
3,950ac9130bafe1fced578bf61d746b047830bfa0,automata/base/exceptions.py,automata/base/exceptions.py,"#!/usr/bin/env python3\n""""""Exception classes s...","#!/usr/bin/env python3\n""""""Exception classes s...","Remove ""validation"" from RejectionException do...","Remove ""validation"" from RejectionException do...",Python,mit,caleb531/automata
4,462ae981ed5b9cc9a8f46e97dfe7908c0827ea64,account_invoice_line_description/res_config.py,account_invoice_line_description/res_config.py,# -*- coding: utf-8 -*-\n#####################...,# -*- coding: utf-8 -*-\n#####################...,"Fix implied_group, it still refers to the old ...","Fix implied_group, it still refers to the old ...",Python,agpl-3.0,"Antiun/account-invoicing,hbrunn/account-invoic..."



Missing values:


,0
commit,0
old_file,0
new_file,0
old_contents,0
new_contents,0
subject,0
message,0
lang,0
license,0
repos,0



Sample commit subjects:


,subject
0,Declare queues when broker is instantiated
1,"Revert ""Fix openweather unit tests"""
2,"Fix % only showing 0 or 100%, everything betwe..."
3,"Remove ""validation"" from RejectionException do..."
4,"Fix implied_group, it still refers to the old ..."
5,Fix interpretation of parameters for names lis...
6,Fix image path in manifest
7,Make debugger record a debug message instead o...
8,Modify the author email address
9,Change the version of the package.


In [ ]:
# Generate Unified Diff Text
def make_unified_diff(row, max_lines=None):
    old_file = row.get("old_file", "old_file.py")
    new_file = row.get("new_file", "new_file.py")
    old_text = row.get("old_contents", "") or ""
    new_text = row.get("new_contents", "") or ""

    diff_lines = list(difflib.unified_diff(
        old_text.splitlines(),
        new_text.splitlines(),
        fromfile=old_file,
        tofile=new_file,
        lineterm=""
    ))

    if max_lines is not None:
        diff_lines = diff_lines[:max_lines]

    return "\n".join(diff_lines)

In [ ]:
df["diff"] = df.apply(make_unified_diff, axis=1)

display(df[["old_file", "new_file", "subject", "message", "diff"]].head())
print(df["diff"].iloc[0][:1500])

,old_file,new_file,subject,message,diff
0,sentry/queue/client.py,sentry/queue/client.py,Declare queues when broker is instantiated,Declare queues when broker is instantiated\n,--- sentry/queue/client.py\n+++ sentry/queue/c...
1,tests/test_openweather.py,tests/test_openweather.py,"Revert ""Fix openweather unit tests""","Revert ""Fix openweather unit tests""\n\nThis re...",--- tests/test_openweather.py\n+++ tests/test_...
2,src/dashboard/src/main/templatetags/percentage.py,src/dashboard/src/main/templatetags/percentage.py,"Fix % only showing 0 or 100%, everything betwe...","Fix % only showing 0 or 100%, everything betwe...",--- src/dashboard/src/main/templatetags/percen...
3,automata/base/exceptions.py,automata/base/exceptions.py,"Remove ""validation"" from RejectionException do...","Remove ""validation"" from RejectionException do...",--- automata/base/exceptions.py\n+++ automata/...
4,account_invoice_line_description/res_config.py,account_invoice_line_description/res_config.py,"Fix implied_group, it still refers to the old ...","Fix implied_group, it still refers to the old ...",--- account_invoice_line_description/res_confi...


--- sentry/queue/client.py
+++ sentry/queue/client.py
@@ -16,6 +16,9 @@
 class Broker(object):
     def __init__(self, config):
         self.connection = BrokerConnection(**config)
+        with producers[self.connection].acquire(block=False) as producer:
+            for queue in task_queues:
+                maybe_declare(queue, producer.channel)
 
     def delay(self, func, *args, **kwargs):
         payload = {
@@ -25,8 +28,6 @@
         }
 
         with producers[self.connection].acquire(block=False) as producer:
-            for queue in task_queues:
-                maybe_declare(queue, producer.channel)
             producer.publish(payload,
                 exchange=task_exchange,
                 serializer="pickle",


In [ ]:
# Basic Data Exploration
df["diff_chars"] = df["diff"].str.len()
df["subject_chars"] = df["subject"].fillna("").str.len()
df["message_chars"] = df["message"].fillna("").str.len()

display(df[["diff_chars", "subject_chars", "message_chars"]].describe())

print("Empty diffs:", (df["diff_chars"] == 0).sum())
print("Empty subjects:", (df["subject"].fillna("").str.strip() == "").sum())
print("Duplicate diffs:", df["diff"].duplicated().sum())
print("Duplicate subjects:", df["subject"].duplicated().sum())

,diff_chars,subject_chars,message_chars
count,56025.000000,56025.000000,56025.000000
mean,781.511183,44.444034,68.125658
std,595.379445,18.049856,87.240436
min,0.000000,15.000000,16.000000
25%,362.000000,33.000000,36.000000
50%,569.000000,41.000000,45.000000
75%,1002.000000,50.000000,62.000000
max,4257.000000,444.000000,3454.000000


Empty diffs: 52
Empty subjects: 0
Duplicate diffs: 1309
Duplicate subjects: 1902


In [ ]:
# Short diffs
display(df.sort_values("diff_chars")[["subject", "diff_chars", "diff"]].head(5))

# Long diffs
display(df.sort_values("diff_chars", ascending=False)[["subject", "diff_chars", "diff"]].head(5))

,subject,diff_chars,diff
24432,Insert empty line at to fix patch.,0,
15070,"Add missing `svn:eol-style : native` prop, whi...",0,
3455,Fix pep8 error with new newline,0,
55396,Fix coding style for travis ci build.,0,
5669,"Add missing `svn:eol-style : native` prop, whi...",0,


,subject,diff_chars,diff
45029,ADD (94) class MockCloudifyContextRelationship...,4257,--- plugin/tests/test_mockcontext.py\n+++ plug...
39037,Add script for copying strings from one projec...,3955,--- java/graveyard/support/scripts/copy-string...
42430,Add script for sending reminders,3891,--- events/management/commands/event_send_remi...
44526,Add base handler for test resources.,3797,--- app/handlers/test_base.py\n+++ app/handler...
44611,Add tests suite for template tags of notificat...,3774,--- apps/notifications/tests/test_templatetags...


## Observations from Data Exploration
1. Mean diff length of 781 is good, but max of 4257 is very large, may need truncation or filtering, else retrieval may degrade
2. 52 Empty diffs, to remove
3. Subject is 44 chars avg, which Message is longer at 68, but can go up to 3k+. Decided to use subject as target output as they are closer to real commit title (Or use both? To decide later)

Decided to filter extreme diff outliers (>3000 chars) and truncate remaining diffs to ~1000 chars.

In [ ]:
# 1. Remove empty diffs
df = df[df["diff"].str.len() > 0]

# 2. Filter extremely long diffs (outliers)
df = df[df["diff_chars"] < 3000]

# 3. Truncate diffs (for consistency)
MAX_DIFF_CHARS = 1000
df["diff"] = df["diff"].str[:MAX_DIFF_CHARS]

# 4. Keep only necessary columns
df = df[["diff", "subject"]].copy()
df.rename(columns={"subject": "commit_message"}, inplace=True)

# 5. Reset index
df.reset_index(drop=True, inplace=True)

print("Final dataset shape:", df.shape)
display(df.head())

Final dataset shape: (55705, 2)


,diff,commit_message
0,--- sentry/queue/client.py\n+++ sentry/queue/c...,Declare queues when broker is instantiated
1,--- tests/test_openweather.py\n+++ tests/test_...,"Revert ""Fix openweather unit tests"""
2,--- src/dashboard/src/main/templatetags/percen...,"Fix % only showing 0 or 100%, everything betwe..."
3,--- automata/base/exceptions.py\n+++ automata/...,"Remove ""validation"" from RejectionException do..."
4,--- account_invoice_line_description/res_confi...,"Fix implied_group, it still refers to the old ..."


In [ ]:
from sklearn.model_selection import train_test_split

# Split processed dataset into retrieval database and held-out evaluation set
train_df, eval_df = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print("Retrieval dataset size:", train_df.shape)
print("Evaluation dataset size:", eval_df.shape)

display(train_df.head())
display(eval_df.head())

Retrieval dataset size: (50134, 2)
Evaluation dataset size: (5571, 2)


,diff,commit_message
0,--- api/core/helpers.py\n+++ api/core/helpers....,Use Mandrill templates to send emails
1,--- examples/00-load/create-tri-surface.py\n++...,Increase point size in example
2,--- tests/test_simple.py\n+++ tests/test_simpl...,Revise tests to reflect removed split
3,--- blaze/tests/test_disk_dimension.py\n+++ bl...,Test dimension perserving for persistence.
4,--- sympy/core/tests/test_cache.py\n+++ sympy/...,Add a test for the @cachit decorator.


,diff,commit_message
0,--- semantic_release/pypi.py\n+++ semantic_rel...,Add dists to twine call
1,--- mmmpaste/db.py\n+++ mmmpaste/db.py\n@@ -30...,Update base 62 id after paste creation.
2,"--- setup.py\n+++ setup.py\n@@ -9,7 +9,7 @@\n ...",Use utf-8 encoding when reading README.rst Avo...
3,--- forum/forms.py\n+++ forum/forms.py\n@@ -15...,Allow Markdown editor to be resized
4,--- script/lib/config.py\n+++ script/lib/confi...,Upgrade libchromiumcontent to fix chromiumviews.


# Scoring/Retrieval
Two options:
1. Fast baseline only (TF-IDF)
2. TF-IDF + embeddings (better but slightly heavier)

### TF-IDF Baseline

In [ ]:
# Build TD-IDF Index
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use the cleaned train_df with columns: diff, commit_message
corpus = train_df["diff"].fillna("").tolist()

vectorizer = TfidfVectorizer(
    max_features=50000,
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(corpus)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (50134, 50000)


In [ ]:
# Retrieval Function
def retrieve_similar_diffs(query_diff, top_k=5):
    query_vec = vectorizer.transform([query_diff])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    results = train_df.iloc[top_indices].copy()
    results["similarity_score"] = scores[top_indices]

    return results[["similarity_score", "commit_message", "diff"]]

In [ ]:
# Test on Existing Diff
# Note: Because we are testing with a diff already inside the dataset, the top result will probably be itself with similarity close to 1.0.
test_idx = 0

query_diff = eval_df.loc[test_idx, "diff"]
true_message = eval_df.loc[test_idx, "commit_message"]

print("Input diff:")
print(query_diff[:1000])

print("\nTrue commit message:")
print(true_message)

print("\nRetrieved similar examples:")
display(retrieve_similar_diffs(query_diff, top_k=5))

Input diff:
--- semantic_release/pypi.py
+++ semantic_release/pypi.py
@@ -16,5 +16,12 @@
     :param password: PyPI account password string
     """
     run('python setup.py {}'.format(dists))
-    run('twine upload -u {} -p {} {}'.format(username, password, '--skip-existing' if skip_existing else ''))
+    run(
+        'twine upload -u {} -p {} {} {}'.format(
+            username,
+            password,
+            '--skip-existing' if skip_existing else '',
+            'dist/*'
+        )
+    )
     run('rm -rf build dist')

True commit message:
Add dists to twine call

Retrieved similar examples:


,similarity_score,commit_message,diff
43833,0.612054,Clean out dist and build before building,--- semantic_release/pypi.py\n+++ semantic_rel...
47134,0.588713,Use new interface for twine,--- semantic_release/pypi.py\n+++ semantic_rel...
20710,0.331452,Fix invoke to fix package finding,"--- tasks.py\n+++ tasks.py\n@@ -10,5 +10,5 @@\..."
34681,0.285616,Remove print. Tag should be made before publish.,"--- setup.py\n+++ setup.py\n@@ -18,7 +18,6 @@\..."
39716,0.276765,Update test after adding cleaning of dist,--- tests/test_pypi.py\n+++ tests/test_pypi.py...


In [ ]:
# Retrieval Function With Exclusion
def retrieve_similar_diffs(query_diff, top_k=5, exclude_idx=None):
    query_vec = vectorizer.transform([query_diff])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    if exclude_idx is not None:
        scores[exclude_idx] = -1

    top_indices = scores.argsort()[::-1][:top_k]

    results = train_df.iloc[top_indices].copy()
    results["similarity_score"] = scores[top_indices]

    return results[["similarity_score", "commit_message", "diff"]]

In [ ]:
# Test Without Self-Match
test_idx = 0

query_diff = eval_df.loc[test_idx, "diff"]
true_message = eval_df.loc[test_idx, "commit_message"]

print("True commit message:")
print(true_message)

print("\nTop retrieved examples excluding itself:")
display(retrieve_similar_diffs(query_diff, top_k=5))

True commit message:
Add dists to twine call

Top retrieved examples excluding itself:


,similarity_score,commit_message,diff
43833,0.612054,Clean out dist and build before building,--- semantic_release/pypi.py\n+++ semantic_rel...
47134,0.588713,Use new interface for twine,--- semantic_release/pypi.py\n+++ semantic_rel...
20710,0.331452,Fix invoke to fix package finding,"--- tasks.py\n+++ tasks.py\n@@ -10,5 +10,5 @@\..."
34681,0.285616,Remove print. Tag should be made before publish.,"--- setup.py\n+++ setup.py\n@@ -18,7 +18,6 @@\..."
39716,0.276765,Update test after adding cleaning of dist,--- tests/test_pypi.py\n+++ tests/test_pypi.py...


### TF-IDF + embeddings

In [ ]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 102.4 MB/s eta 0:00:00


In [ ]:
# Load embedding model
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Prepare diff texts for embedding
diff_texts = train_df["diff"].fillna("").tolist()

print("Number of diffs:", len(diff_texts))
print("Sample diff:")
print(diff_texts[0][:500])

Number of diffs: 50134
Sample diff:
--- api/core/helpers.py
+++ api/core/helpers.py
@@ -14,21 +14,28 @@
     token = get_query_string(user)
     url = base + token
 
-    # TODO: Convert this to an email template
-    if welcome:
-        subject = "Welcome to Voter Engagement"
-    else:
-        subject = "Greetings from Voter Engagement"
-    body = f"Click here to log in: {url}"
-    email = EmailMessage(
-        subject=subject,
-        body=body,
+    message = EmailMessage(
+        subject=None,
         from_email="Citi


In [ ]:
# Encode all diffs
# As we normalized embeddings, inner product search becomes equivalent to cosine similarity
embeddings = model.encode(
    diff_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/784 [00:00<?, ?it/s]

Embedding shape: (50134, 384)


In [ ]:
# Build FAISS index
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 50134


In [ ]:
# Retrieval Function
def retrieve_similar_diffs_embedding(query_diff, top_k=5, exclude_idx=None):
    query_embedding = model.encode(
        [query_diff],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    search_k = top_k + 1 if exclude_idx is not None else top_k
    scores, indices = index.search(query_embedding, search_k)

    scores = scores[0]
    indices = indices[0]

    results = []

    for score, idx in zip(scores, indices):
        if exclude_idx is not None and idx == exclude_idx:
            continue

        results.append({
            "similarity_score": float(score),
            "commit_message": train_df.iloc[idx]["commit_message"],
            "diff": train_df.iloc[idx]["diff"]
        })

        if len(results) == top_k:
            break

    return pd.DataFrame(results)

In [ ]:
# Test embedding retrieval
test_idx = 0

query_diff = eval_df.loc[test_idx, "diff"]
true_message = eval_df.loc[test_idx, "commit_message"]

print("True commit message:")
print(true_message)

print("\nTop retrieved examples using embedding retrieval:")
display(retrieve_similar_diffs_embedding(query_diff, top_k=5))

True commit message:
Add dists to twine call

Top retrieved examples using embedding retrieval:


,similarity_score,commit_message,diff
0,0.905047,Clean out dist and build before building,--- semantic_release/pypi.py\n+++ semantic_rel...
1,0.818041,Use new interface for twine,--- semantic_release/pypi.py\n+++ semantic_rel...
2,0.756800,Use `twine` to deploy Wikked to Pypi.,--- monkeys/release.py\n+++ monkeys/release.py...
3,0.725343,Use twine to upload package,"--- setup.py\n+++ setup.py\n@@ -7,8 +7,11 @@\n..."
4,0.712979,Remove print. Tag should be made before publish.,"--- setup.py\n+++ setup.py\n@@ -18,7 +18,6 @@\..."


In [ ]:
# Compare TF-IDF vs embedding retrieval
test_idx = 0

query_diff = eval_df.loc[test_idx, "diff"]
reference_message = eval_df.loc[test_idx, "commit_message"]

print("Held-out input diff:")
print(query_diff[:1000])

print("\nReference human commit message:")
print(reference_message)

print("\nTF-IDF retrieved examples:")
display(retrieve_similar_diffs(query_diff, top_k=5))

print("\nEmbedding retrieved examples:")
display(retrieve_similar_diffs_embedding(query_diff, top_k=5))

Held-out input diff:
--- semantic_release/pypi.py
+++ semantic_release/pypi.py
@@ -16,5 +16,12 @@
     :param password: PyPI account password string
     """
     run('python setup.py {}'.format(dists))
-    run('twine upload -u {} -p {} {}'.format(username, password, '--skip-existing' if skip_existing else ''))
+    run(
+        'twine upload -u {} -p {} {} {}'.format(
+            username,
+            password,
+            '--skip-existing' if skip_existing else '',
+            'dist/*'
+        )
+    )
     run('rm -rf build dist')

Reference human commit message:
Add dists to twine call

TF-IDF retrieved examples:


,similarity_score,commit_message,diff
43833,0.612054,Clean out dist and build before building,--- semantic_release/pypi.py\n+++ semantic_rel...
47134,0.588713,Use new interface for twine,--- semantic_release/pypi.py\n+++ semantic_rel...
20710,0.331452,Fix invoke to fix package finding,"--- tasks.py\n+++ tasks.py\n@@ -10,5 +10,5 @@\..."
34681,0.285616,Remove print. Tag should be made before publish.,"--- setup.py\n+++ setup.py\n@@ -18,7 +18,6 @@\..."
39716,0.276765,Update test after adding cleaning of dist,--- tests/test_pypi.py\n+++ tests/test_pypi.py...



Embedding retrieved examples:


,similarity_score,commit_message,diff
0,0.905047,Clean out dist and build before building,--- semantic_release/pypi.py\n+++ semantic_rel...
1,0.818041,Use new interface for twine,--- semantic_release/pypi.py\n+++ semantic_rel...
2,0.756800,Use `twine` to deploy Wikked to Pypi.,--- monkeys/release.py\n+++ monkeys/release.py...
3,0.725343,Use twine to upload package,"--- setup.py\n+++ setup.py\n@@ -7,8 +7,11 @@\n..."
4,0.712979,Remove print. Tag should be made before publish.,"--- setup.py\n+++ setup.py\n@@ -18,7 +18,6 @@\..."


In [ ]:
import os
from groq import Groq
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

def generate_smart_commit(current_diff, retriever_function, top_k=3):
    # 1. Get similar examples (this returns a DataFrame)
    results_df = retriever_function(current_diff, top_k=top_k)

    # 2. Build the prompt
    prompt = "Task: Generate a concise, imperative git commit message for the 'Current Diff'.\n"
    prompt += "Use the following examples for style and context:\n\n"

    # Use .iterrows() to get the actual row data
    for i, row in results_df.iterrows():
        # Ensure we use 'commit_message' to match your dataframe column name
        prompt += f"Example {i+1} Diff:\n{row['diff']}\n"
        prompt += f"Example {i+1} Message: {row['commit_message']}\n---\n"

    prompt += f"\nCurrent Diff to process:\n{current_diff}\n\n"
    prompt += "Generated Message:"

    # 3. Call the Groq API
    completion = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.1-8b-instant",
        temperature=0.2,
    )

    return completion.choices[0].message.content, results_df

In [ ]:
# Select a test sample from your dataframe
test_idx = 10
test_diff = df.iloc[test_idx]['diff']
actual_message = df.iloc[test_idx]['commit_message']

# RUN THE PIPELINE
generated_msg, retrieved_df = generate_smart_commit(
    test_diff,
    retrieve_similar_diffs_embedding,
    top_k=3
)

print(f"--- ORIGINAL DIFF (Preview) ---\n{test_diff[:200]}...\n")
print(f"--- GROUND TRUTH ---\n{actual_message}\n")
print(f"--- LLM GENERATED MESSAGE ---\n{generated_msg}\n")

print("--- CONTEXT SAMPLES RETRIEVED ---")
display(retrieved_df[['similarity_score', 'commit_message']])

--- ORIGINAL DIFF (Preview) ---
--- setup.py
+++ setup.py
@@ -24,7 +24,7 @@
         'sortedcontainers>=2.0',
     ],
     extras_require={
-        "minidump": ["minidump==0.0.10"],
+        "minidump": ["minidump>=0.0.10"],
      ...

--- GROUND TRUTH ---
Adjust minidump dependency to >= 0.0.10

--- LLM GENERATED MESSAGE ---
Adjust minidump dependency to >= 0.0.10

--- CONTEXT SAMPLES RETRIEVED ---


,similarity_score,commit_message
0,1.000000,Adjust minidump dependency to >= 0.0.10
1,0.905784,Add minidump and xbe backends to extras_require
2,0.826541,Add optional dependency on arpy


In [ ]:
import os
import json
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)


def build_commit_prompt(current_diff, retrieved_examples_df):
    examples_text = ""

    for example_number, row in enumerate(retrieved_examples_df.itertuples(), start=1):
        examples_text += f"""
Example {example_number}
Similarity score: {getattr(row, "similarity_score", "N/A")}

Diff:
{row.diff}

Human commit message:
{row.commit_message}
---
"""

    prompt = f"""
You are an assistant that writes high-quality Git commit messages.

Task:
Given a current code diff, generate a concise and accurate commit message.

Guidelines:
- Use imperative mood, e.g. "Add", "Fix", "Update", "Refactor"
- Keep the main message under 72 characters if possible
- Focus on the purpose of the change, not every line changed
- Do not mention files unless necessary
- Do not invent behavior that is not supported by the diff
- Use the retrieved examples only as style/context references

Retrieved similar examples:
{examples_text}

Current diff:
{current_diff}

Return your answer as valid JSON with this exact structure:
{{
  "commit_message": "...",
  "commit_type": "feature | fix | refactor | docs | test | style | chore | other",
  "confidence": "high | medium | low",
  "reasoning_summary": "Briefly explain why this commit message fits the diff."
}}
"""

    return prompt.strip()


def generate_smart_commit(current_diff, retriever_function, top_k=5, model_name="llama-3.1-8b-instant"):
    retrieved_examples_df = retriever_function(current_diff, top_k=top_k)

    prompt = build_commit_prompt(
        current_diff=current_diff,
        retrieved_examples_df=retrieved_examples_df
    )

    completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You generate concise, accurate Git commit messages from code diffs."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        model=model_name,
        temperature=0.2,
        response_format={"type": "json_object"}
    )

    raw_output = completion.choices[0].message.content

    try:
        parsed_output = json.loads(raw_output)
    except json.JSONDecodeError:
        parsed_output = {
            "commit_message": raw_output.strip(),
            "commit_type": "other",
            "confidence": "low",
            "reasoning_summary": "Model output was not valid JSON."
        }

    analysis_result = {
        "input_diff": current_diff,
        "generated_commit_message": parsed_output.get("commit_message", ""),
        "commit_type": parsed_output.get("commit_type", "other"),
        "confidence": parsed_output.get("confidence", "low"),
        "reasoning_summary": parsed_output.get("reasoning_summary", ""),
        "retrieved_examples": retrieved_examples_df,
        "prompt": prompt,
        "raw_model_output": raw_output
    }

    return analysis_result

In [ ]:
test_idx = np.random.choice(eval_df.index)

current_diff = eval_df.loc[test_idx, "diff"]
reference_message = eval_df.loc[test_idx, "commit_message"]

result = generate_smart_commit(
    current_diff=current_diff,
    retriever_function=retrieve_similar_diffs_embedding,
    top_k=5
)

print("Input diff:")
print(result["input_diff"])

print("\nReference message:")
print(reference_message)

print("\nGenerated message:")
print(result["generated_commit_message"])

print("\nCommit type:")
print(result["commit_type"])

print("\nConfidence:")
print(result["confidence"])

print("\nReasoning:")
print(result["reasoning_summary"])

print("\nRetrieved examples:")
display(result["retrieved_examples"])

Input diff:
--- linter.py
+++ linter.py
@@ -26,7 +26,9 @@
         # Comments show example output for each line of a Stylint warning
         # /path/to/file/example.styl
         ^.*$\s*
-        # 177:24 colors warning hexidecimal color should be a variable
+        # 177:24  colors  warning  hexidecimal color should be a variable
+        # 177:24  warning  hexidecimal color should be a variable  colors
+        # 177  warning  hexidecimal color should be a variable  colors
         ^(?P<line>\d+):?(?P<col>\d+)?\s*(?P<rule>\w+)?\s*((?P<warning>warning)|(?P<error>error))\s*(?P<message>.+)$\s*
     '''
     multiline = True

Reference message:
Add some more output examples

Generated message:
Update regex to handle variable order changes

Commit type:
refactor

Confidence:
high

Reasoning:
The commit message accurately reflects the purpose of the change, which is to update the regex to handle variable order changes. This is evident from the diff, where the order of variables in the re

,similarity_score,commit_message,diff
0,0.942097,Handle case where rule shows before severity,"--- linter.py\n+++ linter.py\n@@ -27,7 +27,7 @..."
1,0.930155,"Make ""near"" group optional in regex","--- linter.py\n+++ linter.py\n@@ -26,7 +26,7 @..."
2,0.755965,Update regexp due to changes in stylint,"--- linter.py\n+++ linter.py\n@@ -6,7 +6,6 @@\..."
3,0.722207,Fix for pydocstyle > 1.1.1,"--- linter.py\n+++ linter.py\n@@ -28,7 +28,7 @..."
4,0.718527,Update regex to include filename capture group.,"--- linter.py\n+++ linter.py\n@@ -12,7 +12,8 @..."


# Evaluation

In [ ]:
import random
import pandas as pd
import numpy as np

def compute_simple_retrieval_score_summary(eval_df, sample_size=30, top_k=3, random_seed=42):
    random.seed(random_seed)

    sample_size = min(sample_size, len(eval_df))
    sampled_indices = random.sample(range(len(eval_df)), sample_size)

    rows = []

    for eval_idx in sampled_indices:
        query_diff = eval_df.loc[eval_idx, "diff"]
        reference_message = eval_df.loc[eval_idx, "commit_message"]

        tfidf_results = retrieve_similar_diffs(
            query_diff,
            top_k=top_k
        )

        embedding_results = retrieve_similar_diffs_embedding(
            query_diff,
            top_k=top_k
        )

        for method_name, results_df in [
            ("TF-IDF", tfidf_results),
            ("Embedding + FAISS", embedding_results)
        ]:
            scores = results_df["similarity_score"].astype(float).tolist()

            rows.append({
                "eval_index": eval_idx,
                "method": method_name,
                "reference_message": reference_message,
                "top1_similarity": scores[0],
                "top3_average_similarity": np.mean(scores),
                "top1_retrieved_message": results_df.iloc[0]["commit_message"],
                "top2_retrieved_message": results_df.iloc[1]["commit_message"] if len(results_df) > 1 else "",
                "top3_retrieved_message": results_df.iloc[2]["commit_message"] if len(results_df) > 2 else "",
            })

    score_df = pd.DataFrame(rows)

    summary_df = (
        score_df
        .groupby("method")[["top1_similarity", "top3_average_similarity"]]
        .mean()
        .reset_index()
    )

    return score_df, summary_df

In [ ]:
retrieval_score_df, retrieval_summary_df = compute_simple_retrieval_score_summary(
    eval_df=eval_df,
    sample_size=30,
    top_k=3,
    random_seed=42
)

display(retrieval_summary_df)
display(retrieval_score_df.head())

,method,top1_similarity,top3_average_similarity
0,Embedding + FAISS,0.739334,0.692941
1,TF-IDF,0.537639,0.454025


,eval_index,method,reference_message,top1_similarity,top3_average_similarity,top1_retrieved_message,top2_retrieved_message,top3_retrieved_message
0,5238,TF-IDF,Add new urls file for votes,0.580806,0.510347,Fix a couple of redirect URLs,Remove special topics and locations from URLs,Change bill detail page to use session and ide...
1,5238,Embedding + FAISS,Add new urls file for votes,0.772200,0.766944,Add URL scheme for votes app,Fix vote app URL patterns,Add votes URL scheme to main URL scheme
2,912,TF-IDF,Fix in collectd event notifier script.,0.350499,0.323360,Use Memcache to cache tags,Add the sha to the tags in a way that won't ca...,"Add a EXIF extract utility. Later the EXIF ""EX..."
3,912,Embedding + FAISS,Fix in collectd event notifier script.,0.658893,0.632578,Add id tot he beacon event dataset,Add API wrapper to saltapi,Add monitoring state for load average
4,204,TF-IDF,Change inline docs about class fake storage va...,0.251383,0.213309,Create zookeeper node if it doesn't exist,Fix detection of Zookeeper in monasca-setup,Add complete exception unit tests.


In [ ]:
qualitative_sample_df = retrieval_score_df.copy()

qualitative_sample_df = qualitative_sample_df[
    [
        "eval_index",
        "method",
        "reference_message",
        "top1_similarity",
        "top3_average_similarity",
        "top1_retrieved_message",
        "top2_retrieved_message",
        "top3_retrieved_message"
    ]
]

qualitative_sample_df.to_csv(
    "retrieval_qualitative_sample.csv",
    index=False
)

from google.colab import files
files.download("retrieval_qualitative_sample.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>